In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [3]:
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = ImageFolder(
    root="datasets/stop",
    transform=transform
)

print("Classes:", dataset.classes)
print("Class mapping:", dataset.class_to_idx)
print("Total images:", len(dataset))

for class_name in dataset.classes:
    class_index = dataset.class_to_idx[class_name]

    count = sum(
        1 for _, label in dataset.samples
        if label == class_index
    )

    print(class_name, ":", count)

Classes: ['close', 'far', 'none']
Class mapping: {'close': 0, 'far': 1, 'none': 2}
Total images: 534
close : 90
far : 214
none : 230


In [4]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 427
Validation: 107


In [5]:
import torch
import torch.nn as nn

class StopCNN(nn.Module):
    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # 3 classes:
            # 0 = CLOSE
            # 1 = FAR
            # 2 = NONE
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = StopCNN().to(device)

print(model)

StopCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(2, 2))
    (1): ReLU()
    (2): Conv2d(16, 32, kernel_size=(5, 5), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2))
    (5): ReLU()
    (6): AdaptiveAvgPool2d(output_size=(4, 4))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# מספר התמונות בכל class
class_counts = torch.tensor([
    89,   # CLOSE = class 0
    202,  # FAR   = class 1
    85    # NONE  = class 2
], dtype=torch.float32)

# ככל שיש פחות תמונות במחלקה -> משקל גבוה יותר
class_weights = len(dataset) / (3 * class_counts)

class_weights = class_weights.to(device)

print("Class weights:")
print("CLOSE:", class_weights[0].item())
print("FAR  :", class_weights[1].item())
print("NONE :", class_weights[2].item())

# Loss לסיווג 3 מחלקות
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("\nLoss and optimizer ready!")

Class weights:
CLOSE: 1.408239722251892
FAR  : 0.6204620599746704
NONE : 1.474509835243225

Loss and optimizer ready!


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# LOAD OLD TRAINED MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "models/stop_best_model.pth",
        map_location=device
    )
)

print("Old STOP model loaded!")


# ============================================================
# NEW CLASS COUNTS - AUTOMATIC
# ============================================================

class_counts = torch.bincount(
    torch.tensor(dataset.targets),
    minlength=len(dataset.classes)
).float()

print("\nClass counts:")
print("CLOSE:", int(class_counts[0].item()))
print("FAR  :", int(class_counts[1].item()))
print("NONE :", int(class_counts[2].item()))


# ============================================================
# NEW CLASS WEIGHTS
# ============================================================

class_weights = (
    len(dataset) /
    (3 * class_counts)
)

class_weights = class_weights.to(device)

print("\nClass weights:")
print("CLOSE:", class_weights[0].item())
print("FAR  :", class_weights[1].item())
print("NONE :", class_weights[2].item())


# ============================================================
# LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# OPTIMIZER FOR FINE-TUNING
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("\nFine-tuning setup ready!")

Old STOP model loaded!

Class counts:
CLOSE: 90
FAR  : 214
NONE : 230

Class weights:
CLOSE: 1.9777777194976807
FAR  : 0.8317756652832031
NONE : 0.7739130854606628

Fine-tuning setup ready!


In [6]:
import torch
import os

EPOCHS = 20

os.makedirs("models", exist_ok=True)

MODEL_PATH = "models/stop_best_model.pth"

best_val_loss = float("inf")
best_epoch = 0

for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()


    train_loss /= train_total
    train_accuracy = 100.0 * train_correct / train_total


    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()


    val_loss /= val_total
    val_accuracy = 100.0 * val_correct / val_total


    # =========================
    # PRINT
    # =========================
    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )


    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("   -> BEST MODEL SAVED")


print("\nTraining finished!")

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)
print("Saved as:", MODEL_PATH)

Epoch 01/20 | Train Loss: 1.1063 | Train Acc: 49.33% | Val Loss: 1.0997 | Val Acc: 60.53%
   -> BEST MODEL SAVED
Epoch 02/20 | Train Loss: 1.0986 | Train Acc: 46.00% | Val Loss: 1.1017 | Val Acc: 13.16%
Epoch 03/20 | Train Loss: 1.0977 | Train Acc: 28.00% | Val Loss: 1.1052 | Val Acc: 13.16%
Epoch 04/20 | Train Loss: 1.0972 | Train Acc: 25.00% | Val Loss: 1.1065 | Val Acc: 13.16%
Epoch 05/20 | Train Loss: 1.0831 | Train Acc: 36.67% | Val Loss: 1.0704 | Val Acc: 51.32%
   -> BEST MODEL SAVED
Epoch 06/20 | Train Loss: 1.0203 | Train Acc: 52.33% | Val Loss: 1.0401 | Val Acc: 36.84%
   -> BEST MODEL SAVED
Epoch 07/20 | Train Loss: 0.9815 | Train Acc: 49.67% | Val Loss: 1.0121 | Val Acc: 46.05%
   -> BEST MODEL SAVED
Epoch 08/20 | Train Loss: 0.8266 | Train Acc: 67.67% | Val Loss: 0.8365 | Val Acc: 76.32%
   -> BEST MODEL SAVED
Epoch 09/20 | Train Loss: 0.7272 | Train Acc: 70.00% | Val Loss: 0.8650 | Val Acc: 68.42%
Epoch 10/20 | Train Loss: 0.7372 | Train Acc: 69.00% | Val Loss: 0.9666 | V

In [7]:
import torch
import os

# ============================================================
# FINE-TUNING
# ============================================================

EPOCHS = 15

os.makedirs("models", exist_ok=True)

# חשוב: לא דורסים את המודל הישן
MODEL_PATH = "models/stop_best_model_v2.pth"

best_val_loss = float("inf")
best_epoch = 0


for epoch in range(EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() *
            images.size(0)
        )

        _, predicted = torch.max(
            outputs,
            1
        )

        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()


    train_loss /= train_total

    train_accuracy = (
        100.0 *
        train_correct /
        train_total
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() *
                images.size(0)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels.size(0)

            val_correct += (
                predicted == labels
            ).sum().item()


    val_loss /= val_total

    val_accuracy = (
        100.0 *
        val_correct /
        val_total
    )


    # ========================================================
    # RESULTS
    # ========================================================

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2f}%"
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print(
            "   -> BEST V2 MODEL SAVED"
        )


print("\nFine-tuning finished!")

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    best_val_loss
)

print(
    "Saved as:",
    MODEL_PATH
)

Epoch 01/15 | Train Loss: 0.6377 | Train Acc: 73.30% | Val Loss: 0.5157 | Val Acc: 80.37%
   -> BEST V2 MODEL SAVED
Epoch 02/15 | Train Loss: 0.6170 | Train Acc: 75.41% | Val Loss: 0.5032 | Val Acc: 81.31%
   -> BEST V2 MODEL SAVED
Epoch 03/15 | Train Loss: 0.5869 | Train Acc: 78.45% | Val Loss: 0.4906 | Val Acc: 86.92%
   -> BEST V2 MODEL SAVED
Epoch 04/15 | Train Loss: 0.5767 | Train Acc: 78.22% | Val Loss: 0.4636 | Val Acc: 89.72%
   -> BEST V2 MODEL SAVED
Epoch 05/15 | Train Loss: 0.5479 | Train Acc: 81.73% | Val Loss: 0.4506 | Val Acc: 90.65%
   -> BEST V2 MODEL SAVED
Epoch 06/15 | Train Loss: 0.5355 | Train Acc: 81.26% | Val Loss: 0.4432 | Val Acc: 89.72%
   -> BEST V2 MODEL SAVED
Epoch 07/15 | Train Loss: 0.5145 | Train Acc: 81.97% | Val Loss: 0.4178 | Val Acc: 90.65%
   -> BEST V2 MODEL SAVED
Epoch 08/15 | Train Loss: 0.4967 | Train Acc: 83.61% | Val Loss: 0.3959 | Val Acc: 90.65%
   -> BEST V2 MODEL SAVED
Epoch 09/15 | Train Loss: 0.4650 | Train Acc: 85.01% | Val Loss: 0.3919 

In [9]:
# Load best model
model.load_state_dict(
    torch.load(
        "models/stop_best_model_v2.pth",
        map_location=device
    )
)

model.eval()

class_names = dataset.classes

correct_per_class = [0] * len(class_names)
total_per_class = [0] * len(class_names)

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        for label, prediction in zip(labels, predicted):

            label = label.item()
            prediction = prediction.item()

            total_per_class[label] += 1

            if label == prediction:
                correct_per_class[label] += 1


print("Accuracy per class:")
print()

for i, class_name in enumerate(class_names):

    accuracy = (
        100.0 *
        correct_per_class[i] /
        total_per_class[i]
    )

    print(
        f"{class_name.upper():5s}: "
        f"{correct_per_class[i]}/{total_per_class[i]} "
        f"= {accuracy:.2f}%"
    )

Accuracy per class:

CLOSE: 16/17 = 94.12%
FAR  : 38/47 = 80.85%
NONE : 42/43 = 97.67%


In [34]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


import os
import time
import threading
import torch
import ipywidgets as widgets

from IPython.display import display
from jetbot import Robot, bgr8_to_jpeg
from PIL import Image


# ============================================================
# STOP OLD ROBOT
# ============================================================

try:
    robot.stop()
except:
    pass


# ============================================================
# LOAD STOP MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        "models/stop_best_model.pth",
        map_location=device
    )
)

model.eval()

class_names = [
    "CLOSE",   # 0
    "FAR",     # 1
    "NONE"     # 2
]

print("STOP model loaded!")


# ============================================================
# DATASET FOLDERS
# ============================================================

none_dir = "datasets/stop/none"
far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"

os.makedirs(none_dir, exist_ok=True)
os.makedirs(far_dir, exist_ok=True)
os.makedirs(close_dir, exist_ok=True)

CLASS_FOLDERS = {
    "NONE": none_dir,
    "FAR": far_dir,
    "CLOSE": close_dir
}


# ============================================================
# ROBOT
# ============================================================

robot = Robot()

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00

STARTUP_SPEED = 0.18
STARTUP_TIME = 0.15

drive_direction = 0


# ============================================================
# CAMERA VIEW
# ============================================================

camera_view = widgets.Image(
    format="jpeg",
    width=300,
    height=300
)

prediction_label = widgets.Label(
    value="Prediction: ---"
)

confidence_label = widgets.Label(
    value="Confidence: ---"
)


# ============================================================
# DRIVE CONTROLS
# ============================================================

speed_slider = widgets.FloatSlider(
    value=0.09,
    min=0.05,
    max=0.30,
    step=0.01,
    description="Speed:"
)

steering = widgets.FloatSlider(
    value=0.0,
    min=-1.0,
    max=1.0,
    step=0.01,
    description="Steering:"
)

start_button = widgets.Button(
    description="START"
)

stop_button = widgets.Button(
    description="STOP"
)

back_button = widgets.Button(
    description="BACK"
)


# ============================================================
# RECORD SETTINGS
# ============================================================

# לחיצה על RECORD רק מכינה את ההקלטה
record_armed = False

# True רק כשהשמירה באמת פעילה
recording = False

record_class = None

RECORD_INTERVAL = 0.20

current_prediction = None

# Prediction שהיה ברגע שהתחלנו לנסוע ולהקליט
record_start_prediction = None


# ============================================================
# IMAGE HELPERS
# ============================================================

def get_images(folder):

    return sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ])


# ============================================================
# COUNTERS
# ============================================================

none_count = widgets.IntText(
    description="NONE:",
    disabled=True
)

far_count = widgets.IntText(
    description="FAR:",
    disabled=True
)

close_count = widgets.IntText(
    description="CLOSE:",
    disabled=True
)


def update_counts():

    none_count.value = len(
        get_images(none_dir)
    )

    far_count.value = len(
        get_images(far_dir)
    )

    close_count.value = len(
        get_images(close_dir)
    )


update_counts()


# ============================================================
# SAVE ONE IMAGE
# ============================================================

def save_live_image(class_name):

    folder = CLASS_FOLDERS[class_name]

    frame = camera.value

    if frame is None:
        print("No camera frame!")
        return

    filename = (
        str(int(time.time() * 1000))
        + ".jpg"
    )

    path = os.path.join(
        folder,
        filename
    )

    with open(path, "wb") as f:
        f.write(
            bgr8_to_jpeg(frame)
        )

    update_counts()

    print(
        f"Saved as {class_name}:",
        filename
    )


# ============================================================
# MANUAL SAVE BUTTONS
# ============================================================

save_none_button = widgets.Button(
    description="SAVE NONE"
)

save_far_button = widgets.Button(
    description="SAVE FAR"
)

save_close_button = widgets.Button(
    description="SAVE CLOSE"
)


def save_none_live(b):
    save_live_image("NONE")


def save_far_live(b):
    save_live_image("FAR")


def save_close_live(b):
    save_live_image("CLOSE")


save_none_button.on_click(save_none_live)
save_far_button.on_click(save_far_live)
save_close_button.on_click(save_close_live)


# ============================================================
# RECORD BUTTONS
# ============================================================

record_none_button = widgets.Button(
    description="RECORD NONE"
)

record_far_button = widgets.Button(
    description="RECORD FAR"
)

record_close_button = widgets.Button(
    description="RECORD CLOSE"
)

stop_record_button = widgets.Button(
    description="STOP RECORD"
)

record_status = widgets.Label(
    value="Record: OFF"
)


# ============================================================
# ARM RECORD
# ============================================================

def arm_record(class_name):

    global record_armed
    global recording
    global record_class
    global record_start_prediction

    # עדיין לא שומרים שום תמונה
    recording = False

    record_armed = True
    record_class = class_name

    record_start_prediction = None

    record_status.value = (
        f"ARMED: {class_name} | "
        f"Press START"
    )

    print(
        "RECORD ARMED:",
        class_name,
        "- waiting for START"
    )


def record_none(b):
    arm_record("NONE")


def record_far(b):
    arm_record("FAR")


def record_close(b):
    arm_record("CLOSE")


record_none_button.on_click(record_none)
record_far_button.on_click(record_far)
record_close_button.on_click(record_close)


# ============================================================
# RECORD LOOP
# ============================================================

def record_loop():

    global recording

    while recording:

        # שמירה רק בנסיעה קדימה
        if drive_direction != 1:
            time.sleep(0.02)
            continue

        if record_class is None:
            time.sleep(0.02)
            continue

        frame = camera.value

        if frame is not None:

            folder = CLASS_FOLDERS[
                record_class
            ]

            filename = (
                str(int(time.time() * 1000))
                + ".jpg"
            )

            path = os.path.join(
                folder,
                filename
            )

            with open(path, "wb") as f:
                f.write(
                    bgr8_to_jpeg(frame)
                )

            update_counts()

        time.sleep(
            RECORD_INTERVAL
        )


# ============================================================
# START ACTUAL RECORDING
# ============================================================

def start_actual_recording():

    global recording
    global record_start_prediction

    if not record_armed:
        return

    if record_class is None:
        return

    if current_prediction is None:

        print("Waiting for prediction...")
        return

    if recording:
        return

    # זוכרים מה ה-Prediction ברגע ההתחלה
    record_start_prediction = current_prediction

    recording = True

    record_status.value = (
        f"RECORDING: {record_class} | "
        f"Start Prediction: "
        f"{record_start_prediction}"
    )

    threading.Thread(
        target=record_loop,
        daemon=True
    ).start()

    print(
        "RECORD STARTED:",
        record_class,
        "| Prediction:",
        record_start_prediction
    )


# ============================================================
# STOP ACTUAL RECORDING
# ============================================================

def stop_actual_recording(
    keep_armed=True,
    message="RECORD STOPPED"
):

    global recording
    global record_armed
    global record_class
    global record_start_prediction

    recording = False
    record_start_prediction = None

    if keep_armed and record_class is not None:

        record_status.value = (
            f"ARMED: {record_class} | "
            f"Press START"
        )

    else:

        record_armed = False
        record_class = None

        record_status.value = (
            "Record: OFF"
        )

    print(message)


# ============================================================
# MANUAL STOP RECORD
# ============================================================

def stop_record(b=None):

    stop_actual_recording(
        keep_armed=False,
        message="RECORD DISARMED"
    )


stop_record_button.on_click(
    stop_record
)


# ============================================================
# MOTOR CONTROL
# ============================================================

def update_motors(change=None):

    global drive_direction

    if drive_direction == 0:

        robot.stop()
        return

    speed = speed_slider.value
    s = steering.value

    steering_power = (
        s * 0.10
    )

    left_speed = (
        speed + steering_power
    ) * LEFT_GAIN

    right_speed = (
        speed - steering_power
    ) * RIGHT_GAIN

    left_speed *= drive_direction
    right_speed *= drive_direction

    left_speed = max(
        -1.0,
        min(1.0, left_speed)
    )

    right_speed = max(
        -1.0,
        min(1.0, right_speed)
    )

    robot.left_motor.value = (
        left_speed
    )

    robot.right_motor.value = (
        right_speed
    )


# ============================================================
# START DRIVE
# ============================================================

def start_drive(b):

    global drive_direction

    drive_direction = 1

    # Startup boost
    robot.left_motor.value = (
        STARTUP_SPEED * LEFT_GAIN
    )

    robot.right_motor.value = (
        STARTUP_SPEED * RIGHT_GAIN
    )

    time.sleep(
        STARTUP_TIME
    )

    update_motors()

    # רק עכשיו מתחילה ההקלטה
    start_actual_recording()

    print("FORWARD")


# ============================================================
# STOP DRIVE
# ============================================================

def stop_drive(b):

    global drive_direction

    drive_direction = 0

    robot.stop()

    # STOP של הרובוט עוצר מיד גם את השמירה
    if recording:

        stop_actual_recording(
            keep_armed=True,
            message="RECORD STOPPED WITH ROBOT"
        )

    print("STOPPED")


# ============================================================
# BACK
# ============================================================

def back_drive(b):

    global drive_direction

    # אין הקלטה בנסיעה אחורה
    if recording:

        stop_actual_recording(
            keep_armed=True,
            message="RECORD STOPPED - BACKWARD"
        )

    drive_direction = -1

    update_motors()

    print("BACKWARD")


start_button.on_click(
    start_drive
)

stop_button.on_click(
    stop_drive
)

back_button.on_click(
    back_drive
)

speed_slider.observe(
    update_motors,
    names="value"
)

steering.observe(
    update_motors,
    names="value"
)


# ============================================================
# AUTO STOP IMMEDIATELY WHEN PREDICTION CHANGES
# ============================================================

def check_prediction_change(predicted_class):

    global recording
    global record_start_prediction

    if not recording:
        return

    if record_start_prediction is None:
        return

    # עדיין אותו Prediction
    if predicted_class == record_start_prediction:

        record_status.value = (
            f"RECORDING: {record_class} | "
            f"Prediction: {predicted_class}"
        )

        return

    # ========================================================
    # אפילו שינוי אחד בלבד
    # -> מפסיקים לשמור מיד
    # ========================================================

    old_prediction = (
        record_start_prediction
    )

    new_prediction = (
        predicted_class
    )

    stop_actual_recording(
        keep_armed=True,
        message=(
            f"AUTO STOP RECORD: "
            f"{old_prediction} -> "
            f"{new_prediction}"
        )
    )

    record_status.value = (
        f"AUTO STOP: "
        f"{old_prediction} -> "
        f"{new_prediction}"
    )


# ============================================================
# DELETE CLASS SELECTOR
# ============================================================

delete_class = widgets.Dropdown(
    options=[
        "NONE",
        "FAR",
        "CLOSE"
    ],
    value="NONE",
    description="Class:"
)


# ============================================================
# DELETE LAST
# ============================================================

delete_last_button = widgets.Button(
    description="DELETE LAST"
)


def delete_last_image(b):

    if recording:

        print(
            "STOP RECORD before deleting."
        )
        return

    selected_class = (
        delete_class.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    if len(images) == 0:

        print(
            selected_class,
            "folder is empty."
        )

        return

    filename = images[-1]

    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )


delete_last_button.on_click(
    delete_last_image
)


# ============================================================
# DELETE RANGE
# ============================================================

from_box = widgets.IntText(
    value=1,
    description="From:"
)

to_box = widgets.IntText(
    value=1,
    description="To:"
)

delete_range_button = widgets.Button(
    description="DELETE RANGE"
)


def delete_range(b):

    if recording:

        print(
            "STOP RECORD before deleting."
        )
        return

    selected_class = (
        delete_class.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    start = from_box.value
    end = to_box.value

    if start < 1:

        print("From must be >= 1")
        return

    if end < start:

        print("To must be >= From")
        return

    if start > len(images):

        print(
            "Start image does not exist."
        )
        return

    end = min(
        end,
        len(images)
    )

    images_to_delete = (
        images[start - 1:end]
    )

    for filename in images_to_delete:

        os.remove(
            os.path.join(
                folder,
                filename
            )
        )

    update_counts()

    print(
        "Deleted",
        len(images_to_delete),
        "images from",
        selected_class,
        "range",
        start,
        "-",
        end
    )


delete_range_button.on_click(
    delete_range
)


# ============================================================
# DELETE SPECIFIC IMAGE
# ============================================================

image_input = widgets.Text(
    description="Image:",
    placeholder="number or filename"
)

delete_image_button = widgets.Button(
    description="DELETE IMAGE"
)


def delete_specific_image(b):

    if recording:

        print(
            "STOP RECORD before deleting."
        )
        return

    selected_class = (
        delete_class.value
    )

    folder = CLASS_FOLDERS[
        selected_class
    ]

    images = get_images(folder)

    value = (
        image_input.value.strip()
    )

    if value == "":

        print(
            "Enter image number or filename."
        )
        return

    if value.isdigit():

        image_number = int(value)

        if (
            image_number < 1
            or
            image_number > len(images)
        ):

            print(
                "Image number does not exist."
            )
            return

        filename = (
            images[image_number - 1]
        )

    else:

        filename = value

        if filename not in images:

            print(
                "Filename not found:",
                filename
            )
            return

    os.remove(
        os.path.join(
            folder,
            filename
        )
    )

    update_counts()

    print(
        "Deleted:",
        selected_class,
        filename
    )


delete_image_button.on_click(
    delete_specific_image
)


# ============================================================
# LIVE PREDICTION
# ============================================================

def predict_live(change):

    global current_prediction

    frame = camera.value

    if frame is None:
        return

    camera_view.value = (
        bgr8_to_jpeg(frame)
    )

    image = Image.fromarray(
        frame[:, :, ::-1]
    )

    x = transform(image)

    x = x.unsqueeze(0).to(
        device
    )

    with torch.no_grad():

        outputs = model(x)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )

    predicted_class = (
        class_names[
            predicted.item()
        ]
    )

    current_prediction = (
        predicted_class
    )

    confidence_value = (
        confidence.item()
        * 100
    )

    prediction_label.value = (
        f"Prediction: "
        f"{predicted_class}"
    )

    confidence_label.value = (
        f"Confidence: "
        f"{confidence_value:.1f}%"
    )

    # בדיקה בכל Prediction חדש
    check_prediction_change(
        predicted_class
    )


# ============================================================
# CAMERA OBSERVER
# ============================================================

# מסיר callback ישן אם הרצנו את התא בעבר
try:
    camera.unobserve(
        _stop_live_callback,
        names="value"
    )
except:
    pass

_stop_live_callback = predict_live

camera.observe(
    _stop_live_callback,
    names="value"
)


# ============================================================
# DISPLAY
# ============================================================

display(camera_view)

display(
    prediction_label,
    confidence_label
)


print("DRIVE")

display(
    widgets.HBox([
        start_button,
        stop_button,
        back_button
    ])
)

display(speed_slider)
display(steering)


print("SAVE")

display(
    widgets.HBox([
        save_none_button,
        save_far_button,
        save_close_button
    ])
)


print("RECORD")

display(
    widgets.HBox([
        record_none_button,
        record_far_button,
        record_close_button,
        stop_record_button
    ])
)

display(record_status)


print("COUNTS")

display(
    widgets.HBox([
        none_count,
        far_count,
        close_count
    ])
)


print("DELETE")

display(delete_class)

display(delete_last_button)

display(
    widgets.HBox([
        from_box,
        to_box,
        delete_range_button
    ])
)

display(
    widgets.HBox([
        image_input,
        delete_image_button
    ])
)


print("LIVE STOP TEST READY")
print("Prediction change = immediate RECORD stop")

In [37]:
import os
import time
import threading

import torch
import torch.nn as nn

from torchvision import transforms
from PIL import Image

import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, Camera, bgr8_to_jpeg


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)




# ============================================================
# STOP OLD ROBOT
# ============================================================

try:
    robot.stop()
except:
    pass


# ============================================================
# STOP CNN
# ============================================================

class StopCNN(nn.Module):

    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                kernel_size=5,
                stride=2
            ),
            nn.ReLU(),

            nn.Conv2d(
                16,
                32,
                kernel_size=5,
                stride=2
            ),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=2
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 4 * 4,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                128,
                3
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ============================================================
# LOAD STOP MODEL
# ============================================================

model = StopCNN().to(device)

model.load_state_dict(
    torch.load(
        "models/stop_best_model.pth",
        map_location=device
    )
)

model.eval()


class_names = [
    "CLOSE",   # 0
    "FAR",     # 1
    "NONE"     # 2
]

print("STOP model loaded!")


# ============================================================
# TRANSFORM
# ============================================================

transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor()
])


# ============================================================
# DATASET FOLDERS
# ============================================================

none_dir = "datasets/stop/none"
far_dir = "datasets/stop/far"
close_dir = "datasets/stop/close"


os.makedirs(
    none_dir,
    exist_ok=True
)

os.makedirs(
    far_dir,
    exist_ok=True
)

os.makedirs(
    close_dir,
    exist_ok=True
)


CLASS_FOLDERS = {

    "NONE": none_dir,

    "FAR": far_dir,

    "CLOSE": close_dir
}


# ============================================================
# ROBOT
# ============================================================

robot = Robot()


LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00


STARTUP_SPEED = 0.18
STARTUP_TIME = 0.15


# 0  = STOP
# 1  = FORWARD
# -1 = BACKWARD

drive_direction = 0


# ============================================================
# CAMERA VIEW
# ============================================================

camera_view = widgets.Image(

    format="jpeg",

    width=300,

    height=300
)


prediction_label = widgets.Label(

    value="Prediction: ---"
)


confidence_label = widgets.Label(

    value="Confidence: ---"
)


# ============================================================
# DRIVE CONTROLS
# ============================================================

speed_slider = widgets.FloatSlider(

    value=0.09,

    min=0.05,

    max=0.30,

    step=0.01,

    description="Speed:"
)


steering = widgets.FloatSlider(

    value=0.0,

    min=-1.0,

    max=1.0,

    step=0.01,

    description="Steering:"
)


start_button = widgets.Button(

    description="START",

    button_style="success"
)


stop_button = widgets.Button(

    description="STOP",

    button_style="danger"
)


back_button = widgets.Button(

    description="BACK"
)


# ============================================================
# RECORD SETTINGS
# ============================================================

# לחיצה על RECORD רק מכינה את ההקלטה
record_armed = False


# True רק בזמן שמירת תמונות בפועל
recording = False


# NONE / FAR / CLOSE
record_class = None


# שמירת תמונה כל 0.20 שניות
RECORD_INTERVAL = 0.20


# Prediction הנוכחי של המודל
current_prediction = None


# Prediction שהיה ברגע שהתחלנו RECORD
record_start_prediction = None


# ============================================================
# IMAGE HELPERS
# ============================================================

def get_images(folder):

    return sorted([

        f for f in os.listdir(folder)

        if f.lower().endswith(

            (
                ".jpg",
                ".jpeg",
                ".png"
            )
        )
    ])


# ============================================================
# COUNTERS
# ============================================================

none_count = widgets.IntText(

    description="NONE:",

    disabled=True
)


far_count = widgets.IntText(

    description="FAR:",

    disabled=True
)


close_count = widgets.IntText(

    description="CLOSE:",

    disabled=True
)


def update_counts():

    none_count.value = len(
        get_images(
            none_dir
        )
    )

    far_count.value = len(
        get_images(
            far_dir
        )
    )

    close_count.value = len(
        get_images(
            close_dir
        )
    )


update_counts()


# ============================================================
# SAVE ONE IMAGE
# ============================================================

def save_live_image(class_name):

    folder = CLASS_FOLDERS[
        class_name
    ]

    frame = camera.value

    if frame is None:

        print(
            "No camera frame!"
        )

        return


    filename = (

        str(
            int(
                time.time() * 1000
            )
        )

        + ".jpg"
    )


    path = os.path.join(

        folder,

        filename
    )


    with open(
        path,
        "wb"
    ) as f:

        f.write(
            bgr8_to_jpeg(
                frame
            )
        )


    update_counts()


    print(
        f"Saved as {class_name}:",
        filename
    )


# ============================================================
# MANUAL SAVE BUTTONS
# ============================================================

save_none_button = widgets.Button(

    description="SAVE NONE"
)


save_far_button = widgets.Button(

    description="SAVE FAR"
)


save_close_button = widgets.Button(

    description="SAVE CLOSE"
)


def save_none_live(b):

    save_live_image(
        "NONE"
    )


def save_far_live(b):

    save_live_image(
        "FAR"
    )


def save_close_live(b):

    save_live_image(
        "CLOSE"
    )


save_none_button.on_click(
    save_none_live
)

save_far_button.on_click(
    save_far_live
)

save_close_button.on_click(
    save_close_live
)


# ============================================================
# RECORD BUTTONS
# ============================================================

record_none_button = widgets.Button(

    description="RECORD NONE"
)


record_far_button = widgets.Button(

    description="RECORD FAR"
)


record_close_button = widgets.Button(

    description="RECORD CLOSE"
)


stop_record_button = widgets.Button(

    description="STOP RECORD"
)


record_status = widgets.Label(

    value="Record: OFF"
)


# ============================================================
# ARM RECORD
# ============================================================

def arm_record(class_name):

    global record_armed

    global recording

    global record_class

    global record_start_prediction


    # לחיצה על RECORD עדיין לא שומרת תמונות
    recording = False


    record_armed = True

    record_class = class_name


    record_start_prediction = None


    record_status.value = (

        f"ARMED: {class_name} | "

        f"Press START"
    )


    print(

        "RECORD ARMED:",

        class_name,

        "- waiting for START"
    )


def record_none(b):

    arm_record(
        "NONE"
    )


def record_far(b):

    arm_record(
        "FAR"
    )


def record_close(b):

    arm_record(
        "CLOSE"
    )


record_none_button.on_click(
    record_none
)

record_far_button.on_click(
    record_far
)

record_close_button.on_click(
    record_close
)


# ============================================================
# RECORD LOOP
# ============================================================

def record_loop():

    global recording


    while recording:


        # שמירה רק כאשר הרכב נוסע קדימה
        if drive_direction != 1:

            time.sleep(
                0.02
            )

            continue


        if record_class is None:

            time.sleep(
                0.02
            )

            continue


        frame = camera.value


        if frame is not None:


            folder = CLASS_FOLDERS[
                record_class
            ]


            filename = (

                str(
                    int(
                        time.time() * 1000
                    )
                )

                + ".jpg"
            )


            path = os.path.join(

                folder,

                filename
            )


            with open(
                path,
                "wb"
            ) as f:

                f.write(
                    bgr8_to_jpeg(
                        frame
                    )
                )


            update_counts()


        time.sleep(
            RECORD_INTERVAL
        )


# ============================================================
# START ACTUAL RECORDING
# ============================================================

def start_actual_recording():

    global recording

    global record_start_prediction


    if not record_armed:
        return


    if record_class is None:
        return


    if current_prediction is None:

        print(
            "Waiting for prediction..."
        )

        return


    if recording:
        return


    # זוכרים Prediction שהיה ברגע START
    record_start_prediction = (
        current_prediction
    )


    recording = True


    record_status.value = (

        f"RECORDING: {record_class} | "

        f"Start Prediction: "

        f"{record_start_prediction}"
    )


    threading.Thread(

        target=record_loop,

        daemon=True

    ).start()


    print(

        "RECORD STARTED:",

        record_class,

        "| Prediction:",

        record_start_prediction
    )


# ============================================================
# STOP ACTUAL RECORDING
# ============================================================

def stop_actual_recording(

    keep_armed=True,

    message="RECORD STOPPED"
):

    global recording

    global record_armed

    global record_class

    global record_start_prediction


    recording = False


    record_start_prediction = None


    if (
        keep_armed
        and record_class is not None
    ):

        record_status.value = (

            f"ARMED: {record_class} | "

            f"Press START"
        )


    else:

        record_armed = False

        record_class = None


        record_status.value = (
            "Record: OFF"
        )


    print(
        message
    )


# ============================================================
# MANUAL STOP RECORD
# ============================================================

def stop_record(b=None):

    stop_actual_recording(

        keep_armed=False,

        message="RECORD DISARMED"
    )


stop_record_button.on_click(
    stop_record
)


# ============================================================
# MOTOR CONTROL
# ============================================================

def update_motors(change=None):

    global drive_direction


    if drive_direction == 0:

        robot.stop()

        return


    speed = speed_slider.value

    s = steering.value


    steering_power = (

        s * 0.10
    )


    left_speed = (

        speed
        + steering_power

    ) * LEFT_GAIN


    right_speed = (

        speed
        - steering_power

    ) * RIGHT_GAIN


    left_speed *= (
        drive_direction
    )


    right_speed *= (
        drive_direction
    )


    left_speed = max(

        -1.0,

        min(
            1.0,
            left_speed
        )
    )


    right_speed = max(

        -1.0,

        min(
            1.0,
            right_speed
        )
    )


    robot.left_motor.value = (
        left_speed
    )


    robot.right_motor.value = (
        right_speed
    )


# ============================================================
# START DRIVE
# ============================================================

def start_drive(b):

    global drive_direction


    drive_direction = 1


    # Startup boost

    robot.left_motor.value = (

        STARTUP_SPEED
        * LEFT_GAIN
    )


    robot.right_motor.value = (

        STARTUP_SPEED
        * RIGHT_GAIN
    )


    time.sleep(
        STARTUP_TIME
    )


    update_motors()


    # רק אחרי START מתחילה שמירת התמונות
    start_actual_recording()


    print(
        "FORWARD"
    )


# ============================================================
# STOP DRIVE
# ============================================================

def stop_drive(b):

    global drive_direction


    drive_direction = 0


    robot.stop()


    # STOP של הרובוט עוצר גם RECORD

    if recording:

        stop_actual_recording(

            keep_armed=True,

            message=(
                "RECORD STOPPED WITH ROBOT"
            )
        )


    print(
        "STOPPED"
    )


# ============================================================
# BACK
# ============================================================

def back_drive(b):

    global drive_direction


    # אין הקלטה בנסיעה אחורה

    if recording:

        stop_actual_recording(

            keep_armed=True,

            message=(
                "RECORD STOPPED - BACKWARD"
            )
        )


    drive_direction = -1


    update_motors()


    print(
        "BACKWARD"
    )


start_button.on_click(
    start_drive
)

stop_button.on_click(
    stop_drive
)

back_button.on_click(
    back_drive
)


speed_slider.observe(

    update_motors,

    names="value"
)


steering.observe(

    update_motors,

    names="value"
)


# ============================================================
# PREDICTION CHANGE
#
# שינוי אחד בלבד:
#
# 1. RECORD נעצר מיד
# 2. הרכב נעצר מיד
# 3. המחלקה נשארת ARMED
# ============================================================

def check_prediction_change(
    predicted_class
):

    global recording

    global record_start_prediction

    global drive_direction


    # אם אין RECORD פעיל
    # לא עושים שום דבר

    if not recording:
        return


    if record_start_prediction is None:
        return


    # ========================================================
    # PREDICTION עדיין לא השתנה
    # ========================================================

    if (
        predicted_class
        == record_start_prediction
    ):

        record_status.value = (

            f"RECORDING: {record_class} | "

            f"Prediction: {predicted_class}"
        )

        return


    # ========================================================
    # PREDICTION השתנה אפילו פעם אחת
    # ========================================================

    old_prediction = (
        record_start_prediction
    )


    new_prediction = (
        predicted_class
    )


    # ========================================================
    # STOP RECORDING
    # ========================================================

    stop_actual_recording(

        keep_armed=True,

        message=(

            f"AUTO STOP RECORD: "

            f"{old_prediction} -> "

            f"{new_prediction}"
        )
    )


    # ========================================================
    # STOP ROBOT IMMEDIATELY
    # ========================================================

    drive_direction = 0


    robot.stop()


    # ========================================================
    # STATUS
    # ========================================================

    record_status.value = (

        f"AUTO STOP: "

        f"{old_prediction} -> "

        f"{new_prediction} | "

        f"ROBOT STOPPED"
    )


    print(

        "PREDICTION CHANGED:",

        old_prediction,

        "->",

        new_prediction,

        "| RECORD STOPPED | ROBOT STOPPED"
    )


# ============================================================
# DELETE CLASS SELECTOR
# ============================================================

delete_class = widgets.Dropdown(

    options=[
        "NONE",
        "FAR",
        "CLOSE"
    ],

    value="NONE",

    description="Class:"
)


# ============================================================
# DELETE LAST
# ============================================================

delete_last_button = widgets.Button(

    description="DELETE LAST"
)


def delete_last_image(b):


    if recording:

        print(
            "STOP RECORD before deleting."
        )

        return


    selected_class = (
        delete_class.value
    )


    folder = CLASS_FOLDERS[
        selected_class
    ]


    images = get_images(
        folder
    )


    if len(images) == 0:

        print(

            selected_class,

            "folder is empty."
        )

        return


    filename = images[-1]


    os.remove(

        os.path.join(

            folder,

            filename
        )
    )


    update_counts()


    print(

        "Deleted:",

        selected_class,

        filename
    )


delete_last_button.on_click(
    delete_last_image
)


# ============================================================
# DELETE RANGE
# ============================================================

from_box = widgets.IntText(

    value=1,

    description="From:"
)


to_box = widgets.IntText(

    value=1,

    description="To:"
)


delete_range_button = widgets.Button(

    description="DELETE RANGE"
)


def delete_range(b):


    if recording:

        print(
            "STOP RECORD before deleting."
        )

        return


    selected_class = (
        delete_class.value
    )


    folder = CLASS_FOLDERS[
        selected_class
    ]


    images = get_images(
        folder
    )


    start = from_box.value

    end = to_box.value


    if start < 1:

        print(
            "From must be >= 1"
        )

        return


    if end < start:

        print(
            "To must be >= From"
        )

        return


    if start > len(images):

        print(
            "Start image does not exist."
        )

        return


    end = min(

        end,

        len(images)
    )


    images_to_delete = (

        images[
            start - 1:end
        ]
    )


    for filename in images_to_delete:

        os.remove(

            os.path.join(

                folder,

                filename
            )
        )


    update_counts()


    print(

        "Deleted",

        len(images_to_delete),

        "images from",

        selected_class,

        "range",

        start,

        "-",

        end
    )


delete_range_button.on_click(
    delete_range
)


# ============================================================
# DELETE SPECIFIC IMAGE
# ============================================================

image_input = widgets.Text(

    description="Image:",

    placeholder="number or filename"
)


delete_image_button = widgets.Button(

    description="DELETE IMAGE"
)


def delete_specific_image(b):


    if recording:

        print(
            "STOP RECORD before deleting."
        )

        return


    selected_class = (
        delete_class.value
    )


    folder = CLASS_FOLDERS[
        selected_class
    ]


    images = get_images(
        folder
    )


    value = (
        image_input.value.strip()
    )


    if value == "":

        print(
            "Enter image number or filename."
        )

        return


    if value.isdigit():


        image_number = int(
            value
        )


        if (

            image_number < 1

            or

            image_number > len(images)
        ):

            print(
                "Image number does not exist."
            )

            return


        filename = (

            images[
                image_number - 1
            ]
        )


    else:


        filename = value


        if filename not in images:

            print(

                "Filename not found:",

                filename
            )

            return


    os.remove(

        os.path.join(

            folder,

            filename
        )
    )


    update_counts()


    print(

        "Deleted:",

        selected_class,

        filename
    )


delete_image_button.on_click(
    delete_specific_image
)


# ============================================================
# LIVE PREDICTION
# ============================================================

def predict_live(change):

    global current_prediction


    frame = camera.value


    if frame is None:
        return


    # ========================================================
    # LIVE CAMERA
    # ========================================================

    camera_view.value = (

        bgr8_to_jpeg(
            frame
        )
    )


    # ========================================================
    # PREPARE IMAGE
    # ========================================================

    image = Image.fromarray(

        frame[
            :, :, ::-1
        ]
    )


    x = transform(
        image
    )


    x = x.unsqueeze(
        0
    ).to(
        device
    )


    # ========================================================
    # PREDICTION
    # ========================================================

    with torch.no_grad():


        outputs = model(
            x
        )


        probabilities = torch.softmax(

            outputs,

            dim=1
        )


        confidence, predicted = torch.max(

            probabilities,

            dim=1
        )


    predicted_class = (

        class_names[
            predicted.item()
        ]
    )


    current_prediction = (
        predicted_class
    )


    confidence_value = (

        confidence.item()

        * 100
    )


    # ========================================================
    # DISPLAY
    # ========================================================

    prediction_label.value = (

        f"Prediction: "

        f"{predicted_class}"
    )


    confidence_label.value = (

        f"Confidence: "

        f"{confidence_value:.1f}%"
    )


    # ========================================================
    # CHECK IF PREDICTION CHANGED
    # ========================================================

    check_prediction_change(
        predicted_class
    )


# ============================================================
# CAMERA OBSERVER
# ============================================================

# מסיר callback קודם אם התא רץ בעבר

try:

    camera.unobserve(

        _stop_live_callback,

        names="value"
    )

except:
    pass


_stop_live_callback = (
    predict_live
)


camera.observe(

    _stop_live_callback,

    names="value"
)


# ============================================================
# DISPLAY
# ============================================================

display(
    camera_view
)


display(

    prediction_label,

    confidence_label
)


print(
    "DRIVE"
)


display(

    widgets.HBox([

        start_button,

        stop_button,

        back_button
    ])
)


display(
    speed_slider
)


display(
    steering
)


print(
    "SAVE"
)


display(

    widgets.HBox([

        save_none_button,

        save_far_button,

        save_close_button
    ])
)


print(
    "RECORD"
)


display(

    widgets.HBox([

        record_none_button,

        record_far_button,

        record_close_button,

        stop_record_button
    ])
)


display(
    record_status
)


print(
    "COUNTS"
)


display(

    widgets.HBox([

        none_count,

        far_count,

        close_count
    ])
)


print(
    "DELETE"
)


display(
    delete_class
)


display(
    delete_last_button
)


display(

    widgets.HBox([

        from_box,

        to_box,

        delete_range_button
    ])
)


display(

    widgets.HBox([

        image_input,

        delete_image_button
    ])
)


print()
print("LIVE STOP DATASET READY")

print(
    "Prediction change = "
    "RECORD STOP + ROBOT STOP"
)

Device: cuda
STOP model loaded!


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Label(value='Prediction: CLOSE')

Label(value='Confidence: 63.4%')

DRIVE


...

COUNTS


DELETE


Dropdown(description='Class:', options=('NONE', 'FAR', 'CLOSE'), value='NONE')

Button(description='DELETE LAST', style=ButtonStyle())


LIVE STOP DATASET READY
Prediction change = RECORD STOP + ROBOT STOP


In [38]:
camera.stop()
print("Camera stopped")

Camera stopped


In [15]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [17]:
import time
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, bgr8_to_jpeg


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# STEERING MODEL
# ============================================================

class SteeringCNN(nn.Module):

    def __init__(self):
        super(SteeringCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)
        return x.squeeze(1)


# ============================================================
# STOP MODEL
# ============================================================

class StopCNN(nn.Module):

    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# ============================================================
# LOAD MODELS
# ============================================================

steering_model = SteeringCNN().to(device)

steering_model.load_state_dict(
    torch.load(
        "models/clockwise_best_model_3361.pth",
        map_location=device
    )
)

steering_model.eval()


stop_model = StopCNN().to(device)

stop_model.load_state_dict(
    torch.load(
        "models/stop_best_model_v2.pth",
        map_location=device
    )
)

stop_model.eval()

print("Steering model loaded!")
print("STOP V2 model loaded!")


# ============================================================
# TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# ============================================================
# ROBOT
# ============================================================

robot = Robot()


# ============================================================
# SETTINGS
# ============================================================

NORMAL_SPEED = 0.15
FAR_SPEED = 0.10

STOP_TIME = 5.0

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00

MAX_STEERING = 0.15


# ============================================================
# STATES
# ============================================================

auto_running = False

stop_until = 0.0

close_locked = False


# ============================================================
# LIVE CAMERA VIEW
# ============================================================

camera_view = widgets.Image(
    format="jpeg",
    width=320,
    height=320
)


# ============================================================
# LABELS
# ============================================================

steering_label = widgets.Label(
    value="Steering: ---"
)

traffic_label = widgets.Label(
    value="Traffic: ---"
)

confidence_label = widgets.Label(
    value="Confidence: ---"
)

speed_label = widgets.Label(
    value="Speed: 0.00"
)

status_label = widgets.Label(
    value="Status: READY"
)


# ============================================================
# BUTTONS
# ============================================================

start_button = widgets.Button(
    description="START AUTO",
    button_style="success"
)

stop_button = widgets.Button(
    description="STOP",
    button_style="danger"
)


# ============================================================
# DISPLAY
# ============================================================

display(
    widgets.HBox([
        start_button,
        stop_button
    ])
)

display(camera_view)

display(steering_label)
display(traffic_label)
display(confidence_label)
display(speed_label)
display(status_label)


# ============================================================
# MOTOR CONTROL
# ============================================================

def drive(speed, steering):

    steering = max(
        -MAX_STEERING,
        min(MAX_STEERING, steering)
    )

    left_speed = (
        speed + steering
    ) * LEFT_GAIN

    right_speed = (
        speed - steering
    ) * RIGHT_GAIN

    left_speed = max(
        0.0,
        min(1.0, left_speed)
    )

    right_speed = max(
        0.0,
        min(1.0, right_speed)
    )

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed


# ============================================================
# AUTO LOOP
# ============================================================

def auto_drive(change):

    global stop_until
    global close_locked

    frame = camera.value

    if frame is None:
        return


    # --------------------------------------------------------
    # LIVE CAMERA
    # --------------------------------------------------------

    camera_view.value = bgr8_to_jpeg(frame)


    # --------------------------------------------------------
    # IF AUTO OFF -> ONLY SHOW CAMERA
    # --------------------------------------------------------

    if not auto_running:
        return


    # --------------------------------------------------------
    # PREPARE IMAGE
    # --------------------------------------------------------

    image = Image.fromarray(
        frame[:, :, ::-1]
    )

    x = transform(image)

    x = x.unsqueeze(0).to(device)


    # --------------------------------------------------------
    # STEERING PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        steering_value = (
            steering_model(x)
            .item()
        )


    steering_value = max(
        -MAX_STEERING,
        min(MAX_STEERING, steering_value)
    )


    # --------------------------------------------------------
    # TRAFFIC PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        traffic_output = stop_model(x)

        probabilities = torch.softmax(
            traffic_output,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )


    prediction = predicted.item()

    confidence_value = (
        confidence.item() * 100
    )


    # --------------------------------------------------------
    # CLASS MAPPING
    # 0 = CLOSE
    # 1 = FAR
    # 2 = NONE
    # --------------------------------------------------------

    if prediction == 0:
        traffic = "CLOSE"

    elif prediction == 1:
        traffic = "FAR"

    else:
        traffic = "NONE"


    # --------------------------------------------------------
    # UPDATE DISPLAY
    # --------------------------------------------------------

    steering_label.value = (
        f"Steering: {steering_value:.3f}"
    )

    traffic_label.value = (
        f"Traffic: {traffic}"
    )

    confidence_label.value = (
        f"Confidence: {confidence_value:.1f}%"
    )


    # --------------------------------------------------------
    # UNLOCK CLOSE AFTER LEAVING CLOSE AREA
    # --------------------------------------------------------

    if traffic != "CLOSE":
        close_locked = False


    # --------------------------------------------------------
    # NEW CLOSE DETECTION -> STOP FOR 5 SEC
    # --------------------------------------------------------

    if (
        traffic == "CLOSE"
        and not close_locked
        and time.time() >= stop_until
    ):

        robot.stop()

        stop_until = (
            time.time() + STOP_TIME
        )

        close_locked = True

        speed_label.value = "Speed: 0.00"

        status_label.value = (
            "Status: CLOSE -> STOP 5 sec"
        )

        return


    # --------------------------------------------------------
    # CURRENTLY STOPPING
    # --------------------------------------------------------

    if time.time() < stop_until:

        robot.stop()

        remaining = (
            stop_until - time.time()
        )

        speed_label.value = "Speed: 0.00"

        status_label.value = (
            f"Status: STOP {remaining:.1f}s"
        )

        return


    # --------------------------------------------------------
    # FAR -> SLOW DOWN
    # --------------------------------------------------------

    if traffic == "FAR":

        current_speed = FAR_SPEED

        status_label.value = (
            "Status: FAR -> SLOW"
        )


    # --------------------------------------------------------
    # NONE / AFTER STOP
    # --------------------------------------------------------

    else:

        current_speed = NORMAL_SPEED

        status_label.value = (
            "Status: DRIVING"
        )


    # --------------------------------------------------------
    # DRIVE
    # --------------------------------------------------------

    drive(
        current_speed,
        steering_value
    )

    speed_label.value = (
        f"Speed: {current_speed:.2f}"
    )


# ============================================================
# START AUTO
# ============================================================

def start_auto(button):

    global auto_running

    auto_running = True

    status_label.value = "Status: AUTO RUNNING"

    print("AUTO STARTED")


# ============================================================
# STOP AUTO
# ============================================================

def stop_auto(button):

    global auto_running
    global stop_until
    global close_locked

    auto_running = False

    stop_until = 0.0
    close_locked = False

    robot.stop()

    speed_label.value = "Speed: 0.00"
    status_label.value = "Status: STOPPED"

    print("AUTO STOPPED")


start_button.on_click(start_auto)

stop_button.on_click(stop_auto)


# ============================================================
# REMOVE PREVIOUS AUTO CALLBACK
# ============================================================

try:
    camera.unobserve(
        _auto_callback,
        names="value"
    )
except:
    pass


# ============================================================
# CONNECT CAMERA
# ============================================================

_auto_callback = auto_drive

camera.observe(
    _auto_callback,
    names="value"
)


print()
print("AUTO READY")
print("Live camera is ON")
print("Robot will move ONLY after pressing START AUTO")

Device: cuda
Steering model loaded!
STOP V2 model loaded!


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Label(value='Steering: ---')

Label(value='Traffic: ---')

Label(value='Confidence: ---')

Label(value='Speed: 0.00')

Label(value='Status: READY')


AUTO READY
Live camera is ON
Robot will move ONLY after pressing START AUTO


In [26]:
import time
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

from jetbot import Robot, bgr8_to_jpeg


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# STEERING MODEL
# ============================================================

class SteeringCNN(nn.Module):

    def __init__(self):
        super(SteeringCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.regressor(x)

        return x.squeeze(1)


# ============================================================
# STOP MODEL
# ============================================================

class StopCNN(nn.Module):

    def __init__(self):
        super(StopCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=5, stride=2),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=2),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x


# ============================================================
# LOAD STEERING MODEL
# ============================================================

steering_model = SteeringCNN().to(device)

steering_model.load_state_dict(
    torch.load(
        "models/clockwise_best_model_3361.pth",
        map_location=device
    )
)

steering_model.eval()


# ============================================================
# LOAD STOP V2 MODEL
# ============================================================

stop_model = StopCNN().to(device)

stop_model.load_state_dict(
    torch.load(
        "models/stop_best_model_v2.pth",
        map_location=device
    )
)

stop_model.eval()

print("STEERING model loaded!")
print("STOP V2 model loaded!")


# ============================================================
# TRANSFORM
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# ============================================================
# ROBOT
# ============================================================

robot = Robot()


# ============================================================
# SETTINGS
# ============================================================

# -------- STEERING SETTINGS --------

AUTO_SPEED = 0.105

LEFT_MOTOR_GAIN = 1.035
RIGHT_MOTOR_GAIN = 1.00

STEERING_SCALE = 0.10

# המודל החדש למד פניות עד אזור 0.30
MAX_STEERING = 0.30


# -------- TRAFFIC SETTINGS --------

FAR_SPEED = 0.15

STOP_TIME = 5.0

# כדי שפריים בודד שגוי של CLOSE לא יעצור את הרובוט
CLOSE_CONFIRM_FRAMES = 2

# מינימום confidence בשביל לספור CLOSE
CLOSE_CONFIDENCE = 0.070


# ============================================================
# STATES
# ============================================================

auto_running = False

stop_until = 0.0

close_count = 0

close_locked = False


# ============================================================
# LIVE CAMERA
# ============================================================

camera_view = widgets.Image(
    format="jpeg",
    width=320,
    height=320
)


# ============================================================
# INFORMATION
# ============================================================

steering_label = widgets.Label(
    value="Steering: ---"
)

traffic_label = widgets.Label(
    value="Traffic: ---"
)

confidence_label = widgets.Label(
    value="Confidence: ---"
)

speed_label = widgets.Label(
    value="Speed: 0.00"
)

status_label = widgets.Label(
    value="Status: READY"
)


# ============================================================
# BUTTONS
# ============================================================

start_button = widgets.Button(
    description="START AUTO",
    button_style="success"
)

stop_button = widgets.Button(
    description="STOP",
    button_style="danger"
)


# ============================================================
# DISPLAY
# ============================================================

display(
    widgets.HBox([
        start_button,
        stop_button
    ])
)

display(camera_view)

display(steering_label)
display(traffic_label)
display(confidence_label)
display(speed_label)
display(status_label)


# ============================================================
# DRIVE
#
# חשוב:
# זה חישוב ה-STEERING שלך.
# ה-STOP model לא משנה אותו.
# ============================================================

def drive(speed, steering):

    # הגבלה ל +/- 0.30
    steering = max(
        -MAX_STEERING,
        min(MAX_STEERING, steering)
    )

    # בדיוק הסקייל שלך
    steering_power = (
        steering * STEERING_SCALE
    )

    # בדיוק ה-MOTOR GAINS שלך
    left_speed = (
        speed + steering_power
    ) * LEFT_MOTOR_GAIN

    right_speed = (
        speed - steering_power
    ) * RIGHT_MOTOR_GAIN

    # הגנה על ערכי המנועים
    left_speed = max(
        0.0,
        min(1.0, left_speed)
    )

    right_speed = max(
        0.0,
        min(1.0, right_speed)
    )

    robot.left_motor.value = left_speed
    robot.right_motor.value = right_speed


# ============================================================
# AUTO LOOP
# ============================================================

def auto_drive(change):

    global stop_until
    global close_count
    global close_locked

    frame = camera.value

    if frame is None:
        return


    # ========================================================
    # LIVE CAMERA
    # ========================================================

    camera_view.value = bgr8_to_jpeg(frame)


    # המצלמה ממשיכה לעבוד גם כש-AUTO כבוי
    if not auto_running:
        return


    # ========================================================
    # PREPARE IMAGE
    # ========================================================

    image = Image.fromarray(
        frame[:, :, ::-1]
    )

    x = transform(image)

    x = x.unsqueeze(0).to(device)


    # ========================================================
    # STEERING PREDICTION
    # ========================================================

    with torch.no_grad():

        steering_value = (
            steering_model(x).item()
        )


    # אותו MAX_STEERING = 0.30
    steering_value = max(
        -MAX_STEERING,
        min(MAX_STEERING, steering_value)
    )


    # ========================================================
    # STOP MODEL PREDICTION
    # ========================================================

    with torch.no_grad():

        traffic_output = stop_model(x)

        probabilities = torch.softmax(
            traffic_output,
            dim=1
        )

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )


    prediction = predicted.item()

    confidence_value = confidence.item()


    # ========================================================
    # CLASS MAPPING
    #
    # 0 = CLOSE
    # 1 = FAR
    # 2 = NONE
    # ========================================================

    if prediction == 0:

        traffic = "CLOSE"

    elif prediction == 1:

        traffic = "FAR"

    else:

        traffic = "NONE"


    # ========================================================
    # DISPLAY PREDICTIONS
    # ========================================================

    steering_label.value = (
        f"Steering: {steering_value:.3f}"
    )

    traffic_label.value = (
        f"Traffic: {traffic}"
    )

    confidence_label.value = (
        f"Confidence: {confidence_value * 100:.1f}%"
    )


    # ========================================================
    # CURRENTLY INSIDE 5 SECOND STOP
    # ========================================================

    if time.time() < stop_until:

        robot.stop()

        remaining = (
            stop_until - time.time()
        )

        speed_label.value = (
            "Speed: 0.00"
        )

        status_label.value = (
            f"Status: STOP {remaining:.1f}s"
        )

        return


    # ========================================================
    # COUNT CLOSE
    # ========================================================

    if (
        traffic == "CLOSE"
        and confidence_value >= CLOSE_CONFIDENCE
    ):

        close_count += 1

    else:

        close_count = 0


    # ========================================================
    # CLOSE CONFIRMED -> STOP 5 SECONDS
    # ========================================================

    if (
        close_count >= CLOSE_CONFIRM_FRAMES
        and not close_locked
    ):

        robot.stop()

        stop_until = (
            time.time() + STOP_TIME
        )

        close_locked = True

        close_count = 0

        speed_label.value = (
            "Speed: 0.00"
        )

        status_label.value = (
            "Status: CLOSE -> STOP 5 sec"
        )

        return


    # ========================================================
    # RELEASE CLOSE LOCK
    # ========================================================

    if traffic == "NONE":

        close_locked = False


    # ========================================================
    # SELECT SPEED
    # ========================================================

    if traffic == "FAR":

        current_speed = FAR_SPEED

        status_label.value = (
            "Status: FAR -> SPEED 0.08"
        )

    else:

        current_speed = AUTO_SPEED

        status_label.value = (
            "Status: DRIVING"
        )


    # ========================================================
    # DRIVE USING ORIGINAL STEERING SETTINGS
    # ========================================================

    drive(
        current_speed,
        steering_value
    )

    speed_label.value = (
        f"Speed: {current_speed:.2f}"
    )


# ============================================================
# START AUTO
# ============================================================

def start_auto(button):

    global auto_running
    global stop_until
    global close_count
    global close_locked

    stop_until = 0.0
    close_count = 0
    close_locked = False

    auto_running = True

    status_label.value = (
        "Status: AUTO RUNNING"
    )

    print("AUTO STARTED")


# ============================================================
# STOP AUTO
# ============================================================

def stop_auto(button):

    global auto_running
    global stop_until
    global close_count
    global close_locked

    auto_running = False

    stop_until = 0.0
    close_count = 0
    close_locked = False

    robot.stop()

    speed_label.value = (
        "Speed: 0.00"
    )

    status_label.value = (
        "Status: STOPPED"
    )

    print("AUTO STOPPED")


start_button.on_click(start_auto)

stop_button.on_click(stop_auto)


# ============================================================
# REMOVE OLD AUTO CALLBACK
# ============================================================

try:

    camera.unobserve(
        _auto_callback,
        names="value"
    )

except:
    pass


# ============================================================
# CONNECT CAMERA
# ============================================================

_auto_callback = auto_drive

camera.observe(
    _auto_callback,
    names="value"
)


# ============================================================
# READY
# ============================================================

print()
print("AUTO READY")
print("LIVE CAMERA: ON")
print()
print("STEERING SETTINGS:")
print("AUTO_SPEED =", AUTO_SPEED)
print("LEFT_MOTOR_GAIN =", LEFT_MOTOR_GAIN)
print("RIGHT_MOTOR_GAIN =", RIGHT_MOTOR_GAIN)
print("STEERING_SCALE =", STEERING_SCALE)
print("MAX_STEERING =", MAX_STEERING)
print()
print("TRAFFIC SETTINGS:")
print("FAR_SPEED =", FAR_SPEED)
print("STOP_TIME =", STOP_TIME)
print()
print("Robot moves ONLY after START AUTO")

STEERING model loaded!
STOP V2 model loaded!


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Label(value='Steering: ---')

Label(value='Traffic: ---')

Label(value='Confidence: ---')

Label(value='Speed: 0.00')

Label(value='Status: READY')


AUTO READY
LIVE CAMERA: ON

STEERING SETTINGS:
AUTO_SPEED = 0.105
LEFT_MOTOR_GAIN = 1.035
RIGHT_MOTOR_GAIN = 1.0
STEERING_SCALE = 0.1
MAX_STEERING = 0.3

TRAFFIC SETTINGS:
FAR_SPEED = 0.15
STOP_TIME = 5.0

Robot moves ONLY after START AUTO


In [ ]:
camera.stop()
print("Camera stopped")